# A viewport-sized Web Mercator tile grid

Pan or zoom the map to see the XYZ tile boundaries around Baltimore. Each zoom step divides every tile into four. Generate only lines intersecting the visible map extent, so zooming in costs no more than zooming out. The basemap requires internet access; the grid calculation does not.

In [ ]:
import math

import cartopy.crs as ccrs
import geoviews as gv
import holoviews as hv

gv.extension("bokeh")
mercator = ccrs.GOOGLE_MERCATOR
half_world = math.pi * 6378137
width = 768
center = mercator.transform_point(-76.6122, 39.2904, ccrs.PlateCarree())
x0, y0 = center
initial_x = (x0 - 90000, x0 + 90000)
initial_y = (y0 - 60000, y0 + 60000)

Tile edges are integer multiples of the tile width measured from the western or southern world boundary. The displayed zoom below assumes standard 256-pixel tiles. Use the untrimmed viewport width to estimate zoom, then clip the line endpoints to the projected world.

In [ ]:
def grid_lines(x_range, y_range):
    left, right = x_range
    bottom, top = y_range
    span = right - left
    if span <= 0 or top <= bottom:
        return []
    zoom = max(0, min(22, round(math.log2(2 * half_world * width / (256 * span)))))
    step = 2 * half_world / (2 ** zoom)
    left, right = max(left, -half_world), min(right, half_world)
    bottom, top = max(bottom, -half_world), min(top, half_world)
    if left >= right or bottom >= top:
        return []

    def edges(low, high):
        first = math.ceil((low + half_world) / step)
        last = math.floor((high + half_world) / step)
        return (-half_world + i * step for i in range(first, last + 1))

    return ([[(x, bottom), (x, top)] for x in edges(left, right)]
            + [[(left, y), (right, y)] for y in edges(bottom, top)])

In [ ]:
tiles = gv.tile_sources.OSM()
viewport = hv.streams.RangeXY(source=tiles, x_range=initial_x, y_range=initial_y)

def visible_grid(x_range, y_range):
    if not x_range or not y_range or not all(map(math.isfinite, (*x_range, *y_range))):
        x_range, y_range = initial_x, initial_y
    paths = grid_lines(x_range, y_range)
    return gv.Path(paths, crs=mercator)

grid = hv.DynamicMap(visible_grid, streams=[viewport])
(tiles * grid.opts(color="navy", line_width=2)).opts(
    width=width, height=512, xlim=initial_x, ylim=initial_y,
    projection=mercator, active_tools=["wheel_zoom"]
)